# RMV Appointment Checker
This notebook checks for the earliest available dates for given service centers. Please provide your custom URL to start.


In [ ]:
# Replace the URL below with your custom URL
URL = "https://rmvmassdotappt.cxmflow.com/Appointment/Index/[YOUR UNIQUE URL]"

In [ ]:
!pip3 install selenium
!apt-get -qq install -y time

In [ ]:
import time
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC

In [ ]:
from IPython.display import display, Javascript

def alert(message):
    js_code = f'alert("{message}")'
    display(Javascript(js_code))

In [ ]:
def create_driver():
    """Creates a Chrome web driver with specific options."""
    chrome_options = webdriver.ChromeOptions()
    chrome_options.add_argument("--headless")
    chrome_options.add_argument("--disable-gpu")
    chrome_options.add_argument("--no-sandbox")
    chrome_options.add_argument("--window-size=1920,1080")

    driver = webdriver.Chrome(options=chrome_options)
    wait = WebDriverWait(driver, 10)
    return driver, wait

def any_of_conditions(*conditions):
    """Helper function to check for any of multiple expected conditions."""
    def check(driver):
        for condition in conditions:
            try:
                if condition(driver):
                    return True
            except:
                pass
        return False
    return check

def get_service_centers(driver, wait):
    """Extracts and displays a list of service center names from the page.
    Returns a list of selected service centers based on user input."""
    wait.until(EC.presence_of_element_located((By.TAG_NAME, "button")))
    all_buttons = driver.find_elements(By.TAG_NAME, "button")

    centers = [btn.text.split('\n')[0] for btn in all_buttons if btn.text.split('\n')[0] not in ["", "List View", "Map View"]]

    print("\nAvailable Service Centers:")
    for idx, center in enumerate(centers, 1):
        print(f"{idx}. {center}")

    selections = input("\nEnter the numbers of the service centers you want to track (comma separated): ").split(',')
    selected_centers = [centers[int(choice) - 1] for choice in selections]

    return selected_centers

earliest_dates = {}

def check_availability(driver, wait, service_center_name):
    """Check availability for a given service center."""
    all_buttons = driver.find_elements(By.TAG_NAME, "button")
    service_center_button = next((btn for btn in all_buttons if service_center_name in btn.text.split('\n')[0]), None)

    if service_center_button:
        driver.execute_script("arguments[0].click();", service_center_button)
        condition = any_of_conditions(
            EC.presence_of_element_located((By.CLASS_NAME, "DateTimeGrouping-Day")),
            EC.presence_of_element_located((By.XPATH, "//*[contains(., 'There are no times currently available to book')]")),
        )
        wait.until(condition)

        available_dates = driver.find_elements(By.CLASS_NAME, "DateTimeGrouping-Day")
        if available_dates:
            earliest_date_parts = available_dates[0].text.split("\n")
            formatted_date = f"{earliest_date_parts[0]} {earliest_date_parts[1]}"

            # Notify about the found date
            print(f"Earliest available date for {service_center_name}: {formatted_date}", flush=True)

            # Compare the newly found date with the stored one
            if service_center_name not in earliest_dates:
                earliest_dates[service_center_name] = formatted_date
            elif formatted_date < earliest_dates[service_center_name]:
                earliest_dates[service_center_name] = formatted_date
                alert(f"Earlier date found for {service_center_name}: {formatted_date}")
        else:
            print(f"No available dates for {service_center_name}")

        driver.back()
        wait.until(EC.presence_of_element_located((By.TAG_NAME, "button")))


In [ ]:
driver, wait = create_driver()
driver.get(URL)
SERVICE_CENTERS = get_service_centers(driver, wait)


In [ ]:
while True:
    for service_center_name in SERVICE_CENTERS:
        check_availability(driver, wait, service_center_name)

    # Pause for 30 seconds
    print("Pausing for 30 seconds...")
    time.sleep(30)